# WSI Metastasis Segmentation — Kaggle training

**Before running, in the right-hand panel:**

| Setting | Value |
|---|---|
| Session options → Accelerator | **GPU P100** (preferred) or **GPU T4 x2** |
| Session options → Internet | **On** (needs phone verification) |

Then edit **one line** in cell 1 and choose *Run All*.

---

### Where the data comes from

Worked out automatically, in order of preference:

1. A Kaggle Dataset attached under *Input* — instant, no transfer.
2. Already downloaded in this session — reused.
3. Your Google Drive folder — pulled over Kaggle's connection, which is far
   faster than uploading from home.

After a Drive download it offers to publish the data as a Kaggle Dataset,
so every later session takes route 1 and transfers nothing.

### Sessions

Run the short cells here in the editor. For the long training run use
**Save Version → Save & Run All**, which executes server-side for up to
12 hours with the browser closed. The interactive editor disconnects after
roughly 90 idle minutes.


## 1. Configuration — the only cell you edit


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# REQUIRED on the first run only. Share the Drive folder holding
# shards/ + manifest.parquet + export_info.json as
# 'Anyone with the link → Viewer', and paste the link here.
# Leave it empty once the Kaggle Dataset exists.
DRIVE_FOLDER_URL = ''   # e.g. 'https://drive.google.com/drive/folders/1AbC...'

# Already correct — change only if you forked the repository.
REPO = 'https://github.com/arcii99/medical-image-segmentation-wsi.git'

# Training. Early stopping (patience 8) usually halts well before this.
MAX_EPOCHS      = 40
MAX_VAL_BATCHES = 150   # full validation is ~700 batches, ~10 min/epoch
WORKERS         = 2     # Kaggle VMs have 2-4 cores

# Optional: publish the data as a Kaggle Dataset after a Drive download,
# so later sessions skip the transfer. Needs Add-ons → Secrets with
# KAGGLE_USERNAME and KAGGLE_KEY. Skipped silently if absent.
PUBLISH_DATASET = True
DATASET_SLUG    = 'archittiwari99/camelyon16-patchset-512'

# Fallback if Drive rate-limits gdown. Put the contents of
# ~/.config/rclone/rclone.conf into a Kaggle Secret named RCLONE_CONF
# (keep the notebook private -- it holds an OAuth token).
RCLONE_PATH = 'gdrive:wsi/patchset'
# ─────────────────────────────────────────────────────────────────────

WORK, TEMP = '/kaggle/working', '/kaggle/temp'
PATCHSET   = f'{TEMP}/patchset'
CKPT_DIR   = f'{WORK}/ckpt'
import os
os.makedirs(TEMP, exist_ok=True)
print('config loaded')


## 2. Environment

Fails immediately and clearly if the GPU or internet is missing, rather
than later in a way that looks like a different problem.


In [ ]:
import os, socket, subprocess, sys
import torch

print('torch      ', torch.__version__)
assert torch.cuda.is_available(), (
    'No GPU. Session options -> Accelerator -> GPU P100 or GPU T4 x2')
cap = torch.cuda.get_device_capability(0)
print('gpu        ', torch.cuda.get_device_name(0), '(sm_%d%d)' % cap)
print('cpu cores  ', os.cpu_count())

try:
    socket.create_connection(('pypi.org', 443), timeout=8).close()
    print('internet    on')
except OSError:
    raise SystemExit('No internet. Session options -> Internet -> On '
                     '(requires phone verification).')

if cap[0] < 8:
    print()
    print('note: no native bfloat16 on this GPU, so training will select')
    print('      float16 + GradScaler automatically.')


## 3. Code


In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/wsi-metastasis-seg
!git clone -q $REPO /kaggle/working/wsi-metastasis-seg
%cd /kaggle/working/wsi-metastasis-seg
!pip install -q -e '.[dev]' segmentation-models-pytorch 2>&1 | tail -2

!python scripts/version.py || true
!python scripts/check_repo.py || true


In [ ]:
# Refuse to continue on a repository that predates the fixes this
# notebook depends on. Both would otherwise fail in confusing ways: the
# worker hook crashes every DataLoader worker, and the bfloat16 default
# runs ~6x slow on a pre-Ampere GPU.
import pathlib

need = [('src/data/dataset.py', 'hasattr(ds, "_readers")',
         'worker_init must work with the patchset backend (BUG-030)'),
        ('scripts/03_train.py', 'has no native bfloat16',
         'autocast dtype must follow GPU capability (BUG-031)'),
        ('src/data/patchset.py', 'PatchSetDataset',
         'patchset dataset backend')]
missing = [why for f, needle, why in need
           if not pathlib.Path(f).exists()
           or needle not in pathlib.Path(f).read_text()]
if missing:
    raise SystemExit('Repository is out of date:\n  - '
                     + '\n  - '.join(missing)
                     + '\n\nPush the current version to GitHub, then re-run.')
print('repository has the required fixes')


## 4. Data

Extracted to `/kaggle/temp`, not `/kaggle/working`. Twelve gigabytes would
consume the working quota the checkpoints need, and re-extracting costs
about five minutes.


In [ ]:
import glob, os, shutil, subprocess, sys, time

def find_source():
    """Locate the patch set, whatever shape it arrived in.

    Kaggle datasets land in one of three layouts, and which one you get
    depends on how the dataset was uploaded. All three must work:

      A  <root>/shards/*.tar        directory uploaded intact
      B  <root>/*.tar               archives uploaded as loose files
      C  <root>/*.jpg + manifest    Kaggle expanded the archives itself

    C is the best case -- nothing to extract at all.
    """
    roots = sorted(glob.glob('/kaggle/input/*'))

    # 1. Prefer pre-extracted files, checked across ALL attached datasets
    #    first. A patchset uploaded with both loose files AND the .tar
    #    archives has two valid roots; extracting the tars when the jpgs
    #    are already on disk is wasted time and, on Kaggle's small /tmp,
    #    fails partway.
    for r in roots:
        jpg = glob.glob(f'{r}/**/*.jpg', recursive=True)
        png = glob.glob(f'{r}/**/*.png', recursive=True)
        man = glob.glob(f'{r}/**/manifest.parquet', recursive=True)
        if man and len(jpg) > 100 and len(jpg) == len(png):
            return os.path.dirname(jpg[0]), 'kaggle dataset (pre-extracted)'

    # 2. Otherwise fall back to archives.
    for r in roots:
        if glob.glob(f'{r}/**/*.tar', recursive=True):
            return r, 'kaggle dataset (archives)'

    if glob.glob(f'{TEMP}/dl/**/*.tar', recursive=True):
        return f'{TEMP}/dl', 'already downloaded this session'
    return None, None

SRC, HOW = find_source()

if SRC is None:
    if not DRIVE_FOLDER_URL:
        raise SystemExit(
            'No data found.\n'
            'Either attach the Kaggle Dataset under Input, or set '
            'DRIVE_FOLDER_URL in cell 1.')

    DL = f'{TEMP}/dl'
    os.makedirs(DL, exist_ok=True)

    def n_tars():
        return len(glob.glob(f'{DL}/**/*.tar', recursive=True))

    # Drive limits how often a public link may be fetched, and gdown
    # aborts the whole folder when one file trips that. Retrying is not
    # wasteful: gdown skips files already on disk, so each attempt only
    # picks up what is still missing.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'],
                   check=True)
    t0, stalled = time.time(), 0
    for attempt in range(1, 7):
        before = n_tars()
        print(f'--- gdown attempt {attempt} (have {before} shards) ---', flush=True)
        subprocess.run(['gdown', '--folder', DRIVE_FOLDER_URL,
                        '-O', DL, '--remaining-ok'], check=False)
        after = n_tars()
        print(f'    now {after} shards after {time.time()-t0:.0f}s', flush=True)
        if after == before:
            stalled += 1
            if stalled >= 2:
                print('    no progress twice in a row - stopping retries')
                break
            time.sleep(30)
        else:
            stalled = 0

    # Fallback: rclone talks to the Drive API with your own OAuth token,
    # so it backs off properly instead of scraping public links. Put the
    # contents of ~/.config/rclone/rclone.conf in a Kaggle Secret named
    # RCLONE_CONF, and set RCLONE_PATH to e.g. gdrive:wsi/patchset.
    # Trigger on INCOMPLETE, not just empty. gdown often returns a few
    # shards before Drive refuses the rest, and 'not zero' is not the
    # same as 'done'. manifest.parquet is small and downloads reliably,
    # so it can say how many shards there should be.
    expected = None
    man = glob.glob(f'{DL}/**/manifest.parquet', recursive=True)
    if man:
        import pandas as _pd
        mm = _pd.read_parquet(man[0])
        if 'shard' in mm.columns:
            expected = int(mm.shard.nunique())
            print(f'manifest expects {expected} shards; have {n_tars()}')
    incomplete = n_tars() == 0 or (expected is not None and n_tars() < expected)
    if incomplete or os.environ.get('FORCE_RCLONE'):
        try:
            from kaggle_secrets import UserSecretsClient
            conf = UserSecretsClient().get_secret('RCLONE_CONF')
            os.makedirs('/root/.config/rclone', exist_ok=True)
            open('/root/.config/rclone/rclone.conf', 'w').write(conf)
            subprocess.run('curl -s https://rclone.org/install.sh | bash',
                           shell=True, check=False)
            print('--- rclone fallback ---', flush=True)
            subprocess.run(['rclone', 'copy', RCLONE_PATH, DL,
                            '-P', '--transfers', '4'], check=True)
        except Exception as e:
            print(f'rclone fallback unavailable ({type(e).__name__}: {e})')

    # gdown nests the folder one level deep
    if not glob.glob(f'{DL}/shards/*.tar'):
        inner = glob.glob(f'{DL}/*/shards')
        if inner:
            base = os.path.dirname(inner[0])
            for f in glob.glob(f'{base}/*'):
                shutil.move(f, f'{DL}/')
    if not glob.glob(f'{DL}/shards/*.tar'):
        raise SystemExit('nothing downloaded - see the errors above')
    SRC, HOW = DL, 'google drive'
    print(f'downloaded in {time.time()-t0:.0f}s')

print(f'source: {SRC}   ({HOW})')
print(sorted(os.listdir(SRC)))


In [ ]:
import os, glob, shutil, subprocess, time

os.makedirs(PATCHSET, exist_ok=True)

if HOW and 'pre-extracted' in HOW:
    # Kaggle already expanded the archives. /kaggle/input is read-only,
    # but a symlink costs no disk and no time.
    link = f'{PATCHSET}/files'
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(SRC, link)
    for name in ('manifest.parquet', 'export_info.json'):
        hit = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
        if hit:
            shutil.copy2(hit[0], f'{PATCHSET}/{name}')
    import pandas as _pd
    _m = _pd.read_parquet(f'{PATCHSET}/manifest.parquet')
    _jpg = len(glob.glob(f'{SRC}/**/*.jpg', recursive=True))
    assert _jpg == len(_m), (
        f'pre-extracted set has {_jpg} jpg but manifest lists {len(_m)} '
        'patches -- dataset is incomplete')
    print(f'linked {SRC} -> {link}   ({_jpg:,} patches, no extraction)')
else:
    os.makedirs(f'{PATCHSET}/files', exist_ok=True)
    for name in ('manifest.parquet', 'export_info.json'):
        hit = glob.glob(f'{SRC}/**/{name}', recursive=True)
        if hit:
            shutil.copy2(hit[0], f'{PATCHSET}/{name}')

    shards = sorted(glob.glob(f'{SRC}/**/*.tar', recursive=True))
    assert shards, f'no .tar archives under {SRC}'
    size = sum(os.path.getsize(x) for x in shards) / 2**30
    print(f'{len(shards)} shards, {size:.1f} GiB')
    t0 = time.time()
    for i, x in enumerate(shards, 1):
        subprocess.run(['tar', 'xf', x, '-C', f'{PATCHSET}/files'], check=True)
        print(f'  {i}/{len(shards)}  {os.path.basename(x)}'
              f'  ({time.time()-t0:.0f}s)', flush=True)

n_files = len(os.listdir(f'{PATCHSET}/files'))
print(f'\nextracted {n_files:,} files')


In [ ]:
# Verify against the manifest. This catches anything lost in transfer:
# two files per patch, a .jpg and a .png.
import pandas as pd

m = pd.read_parquet(f'{PATCHSET}/manifest.parquet')
expected = len(m) * 2
print(f'manifest rows  {len(m):,}')
print(f'files expected {expected:,}')
print(f'files present  {n_files:,}')
assert n_files == expected, f'MISSING {expected - n_files:,} files'
print('COMPLETE')
print(m.split.value_counts().to_dict())
print(open(f'{PATCHSET}/export_info.json').read())


### Publish as a Kaggle Dataset (first run only)

Runs only if the data came from Drive and credentials exist in
*Add-ons → Secrets* (`KAGGLE_USERNAME`, `KAGGLE_KEY`). This upload is
Kaggle-to-Kaggle, so it takes minutes rather than hours. Later sessions
then attach the dataset and transfer nothing.

Safe to skip — it changes nothing about training.


In [ ]:
if PUBLISH_DATASET and HOW == 'google drive':
    try:
        import json as _json
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        os.environ['KAGGLE_USERNAME'] = sec.get_secret('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = sec.get_secret('KAGGLE_KEY')
        _json.dump({'title': 'CAMELYON16 patchset 512px',
                    'id': DATASET_SLUG,
                    'licenses': [{'name': 'CC0-1.0'}]},
                   open(f'{SRC}/dataset-metadata.json', 'w'))
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', SRC],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)
        print('\nNext session: Input -> Add Input -> search for',
              DATASET_SLUG.split('/')[-1])
    except Exception as e:
        print(f'skipped ({type(e).__name__}: {e})')
        print('Add KAGGLE_USERNAME and KAGGLE_KEY under Add-ons -> Secrets',
              'to enable this.')
else:
    print('nothing to publish')


## 5. Dry run — seconds

One forward and backward pass. Confirms shapes match, the loss is finite,
gradients flow, and the mask is binary. This has caught two bugs in this
project, so it is not ceremonial.


In [ ]:
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=$PATCHSET \
  --set hw.dataloader_workers=$WORKERS --dry-run


## 6. Gate V6.2 — can it memorise one batch?

A model that cannot fit a single batch has a wiring defect no amount of
data will fix. Expect `GATE V6.2 ... PASS` with dice ≥ 0.97.

Also read **margin** on the validation line — the gap between mean
predicted probability on tumour and on background. A high dice with a
margin under 0.1 means the shape is learned but the calibration is not.


In [ ]:
!python scripts/03_train.py \
  --config configs/base.yaml configs/data_camelyon16.yaml \
           configs/model_unet_effb0.yaml configs/patchset.yaml \
  --set data.patchset_root=$PATCHSET \
  --set hw.dataloader_workers=$WORKERS \
  --set train.overfit_batches=1 --set train.max_steps=200 \
  --set data.augment=false --set paths.ckpt=$TEMP/smoke


## 7. Train

**Resumes by itself.** If a previous notebook version is attached under
*Input*, the cell below copies its newest checkpoint into
`/kaggle/working` and continues from that epoch. Otherwise it starts
fresh. Nothing to edit either way.

Run this through **Save Version → Save & Run All**.


In [ ]:
import glob, os, shutil, torch

os.makedirs(CKPT_DIR, exist_ok=True)
RESUME, RUN_ID = None, None

# Checkpoints can be in two places and the search must cover both:
#   /kaggle/working/ckpt   this session, e.g. after a re-run
#   /kaggle/input/.../ckpt a previous version attached under Input
# Looking only in /kaggle/input meant a same-session re-run silently
# restarted from epoch 0.
here = glob.glob(f'{CKPT_DIR}/*/last.pt')
there = (glob.glob('/kaggle/input/*/ckpt/*/last.pt')
         + glob.glob('/kaggle/input/*/*/ckpt/*/last.pt'))
prev = sorted(here + there, key=os.path.getmtime)

if prev:
    src_dir = os.path.dirname(prev[-1])
    RUN_ID = os.path.basename(src_dir)
    dst_dir = f'{CKPT_DIR}/{RUN_ID}'
    if os.path.abspath(src_dir) != os.path.abspath(dst_dir):
        os.makedirs(dst_dir, exist_ok=True)
        for f in glob.glob(f'{src_dir}/*'):
            shutil.copy2(f, dst_dir)
    RESUME = f'{dst_dir}/last.pt'
    done = torch.load(RESUME, map_location='cpu')['epoch']
    print(f'resuming {RUN_ID} from epoch {done + 1}',
          '(this session)' if prev[-1] in here else '(attached version)')
else:
    print('NO CHECKPOINT FOUND - starting from epoch 0.')
    print()
    print('If you expected to resume: Kaggle wipes /kaggle/working between')
    print('sessions. The previous run must be attached explicitly --')
    print('  Input -> Add Input -> Notebook Output -> your previous version')
    print('Without that there is nothing on disk to continue from.')
    print('no previous checkpoint found - starting from scratch')


In [ ]:
cmd = ('python scripts/03_train.py'
       ' --config configs/base.yaml configs/data_camelyon16.yaml'
       ' configs/model_unet_effb0.yaml configs/patchset.yaml'
       f' --set data.patchset_root={PATCHSET}'
       f' --set hw.dataloader_workers={WORKERS}'
       f' --set paths.ckpt={CKPT_DIR}'
       f' --set train.max_epochs={MAX_EPOCHS}'
       f' --set train.max_val_batches={MAX_VAL_BATCHES}')
if RESUME:
    cmd += f' --resume {RESUME}'
print(cmd.replace(' --set', '\n  --set').replace(' --resume', '\n  --resume'))
print()
!{cmd}


## 8. Checkpoint check (gate V7.2)

A checkpoint without a fitted threshold cannot be evaluated: inference
uses that one number, and silently defaulting it to 0.5 is a 5–10 point
error.


In [ ]:
import glob, os, torch

best = sorted(glob.glob(f'{CKPT_DIR}/*/best.pt'), key=os.path.getmtime)
assert best, 'no best.pt - did training run?'
BEST = best[-1]
RUN_ID = os.path.basename(os.path.dirname(BEST))
c = torch.load(BEST, map_location='cpu')
req = {'state_dict', 'ema_state_dict', 'optimizer', 'epoch', 'threshold',
       'resolved_config', 'git_sha', 'dirty', 'pip_freeze', 'index_hash',
       'seed'}
missing = req - set(c)
print('MISSING', missing) if missing else print('COMPLETE')
print(f'run {RUN_ID} | epoch {c["epoch"]} | tau {c["threshold"]:.2f}'
      f' | dirty {c["dirty"]}')
assert 0.30 <= c['threshold'] <= 0.70, 'tau outside the expected band (ADR-006)'


---
## 9. Inference and evaluation

Run these **after** training has converged, ideally in a fresh session.

Whole slides are needed here, but only the nine in the test split — and
they come straight from the public S3 bucket, not from your machine.
About 16 GB into scratch.

Requires the index dataset (`slides.parquet` + `tissue/`) attached under
*Input*.


In [ ]:
import glob, os, pandas as pd

meta = (glob.glob('/kaggle/input/*/slides.parquet')
        + glob.glob('/kaggle/input/*/*/slides.parquet'))
assert meta, 'attach the index dataset (slides.parquet + tissue/) under Input'
META = os.path.dirname(meta[0])
sl = pd.read_parquet(f'{META}/slides.parquet')
test = sl[sl.split == 'test'].slide_id.tolist()
print(META, '|', len(test), 'test slides:', test)


In [ ]:
!pip install -q awscli
!mkdir -p $TEMP/slides
for s in test:
    !aws s3 cp --no-sign-request --quiet --region us-west-2 \
        s3://camelyon-dataset/CAMELYON16/images/{s}.tif $TEMP/slides/{s}.tif
    !aws s3 cp --no-sign-request --quiet --region us-west-2 \
        s3://camelyon-dataset/CAMELYON16/annotations/{s}.xml $TEMP/slides/{s}.xml || true
!du -sh $TEMP/slides


In [ ]:
for s in test:
    !python scripts/04_infer_slide.py --ckpt $BEST \
      --slide $TEMP/slides/{s}.tif \
      --set paths.tissue=$META/tissue \
      --set paths.artifacts=$WORK/artifacts


In [ ]:
!python scripts/05_evaluate.py --run-id $RUN_ID --split test \
  --set paths.artifacts=$WORK/artifacts

import json
print(json.dumps(json.load(open(
    f'{WORK}/artifacts/reports/{RUN_ID}/metrics.json')), indent=2))


---
**For the write-up:** state that the model was trained on an exported
patch set — crops fixed at export time rather than jittered every epoch,
and no hard-negative mining. That is a real difference from the documented
procedure, and anyone comparing against published CAMELYON16 numbers
should know.
